# 01 - Data Exploration

Goal: inspect the IoT-23 CSV safely before preprocessing and graph construction.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "iot23_combined_new.csv"

print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Data exists:", DATA_PATH.exists())

## Read a small sample

The full CSV is large, so first we load only a small sample.

In [ ]:
df_sample = pd.read_csv(DATA_PATH, nrows=10_000)

# The Kaggle CSV contains an old saved index column.
df_sample = df_sample.drop(columns=["Unnamed: 0"], errors="ignore")

display(df_sample.head())
print("Shape:", df_sample.shape)

In [ ]:
df_sample.info()

## Missing values and placeholder values

In [ ]:
missing_report = (
    df_sample.replace("-", np.nan)
    .isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_ratio")
)

missing_report

## Label distribution on the sample

In [ ]:
sample_label_counts = df_sample["label"].value_counts()
sample_label_ratio = df_sample["label"].value_counts(normalize=True)

pd.DataFrame({
    "count": sample_label_counts,
    "ratio": sample_label_ratio,
})

## Label distribution on the full file

This reads only selected columns in chunks, so it is safer than loading the full CSV.

In [ ]:
chunk_size = 500_000
total_rows = 0
label_counts = pd.Series(dtype="int64")
proto_counts = pd.Series(dtype="int64")
service_counts = pd.Series(dtype="int64")
conn_state_counts = pd.Series(dtype="int64")

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["label", "proto", "service", "conn_state"],
    chunksize=chunk_size,
):
    total_rows += len(chunk)
    label_counts = label_counts.add(chunk["label"].value_counts(), fill_value=0)
    proto_counts = proto_counts.add(chunk["proto"].value_counts(), fill_value=0)
    service_counts = service_counts.add(chunk["service"].value_counts(), fill_value=0)
    conn_state_counts = conn_state_counts.add(chunk["conn_state"].value_counts(), fill_value=0)

label_counts = label_counts.astype(int).sort_values(ascending=False)

label_summary = pd.DataFrame({
    "count": label_counts,
    "ratio": label_counts / total_rows,
})

print("Total rows:", total_rows)
label_summary

In [ ]:
print("Protocol distribution")
display(proto_counts.astype(int).sort_values(ascending=False).to_frame("count"))

print("Top services")
display(service_counts.astype(int).sort_values(ascending=False).head(10).to_frame("count"))

print("Top connection states")
display(conn_state_counts.astype(int).sort_values(ascending=False).head(10).to_frame("count"))

## First preprocessing decisions

These are initial decisions for the next notebook/script.

In [ ]:
target_col = "label"

drop_cols = [
    "uid",          # unique flow id, not useful as a model feature
    "local_orig",   # all '-' in this dataset
    "local_resp",   # all '-' in this dataset
]

ip_cols = ["id.orig_h", "id.resp_h"]
categorical_cols = ["proto", "service", "conn_state", "history"]

numeric_cols = [
    col
    for col in df_sample.columns
    if col not in drop_cols + ip_cols + categorical_cols + [target_col]
]

print("Target:", target_col)
print("Drop columns:", drop_cols)
print("IP columns:", ip_cols)
print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)